In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
import joblib
from sklearn.compose import ColumnTransformer

In [2]:
df = pd.read_csv('cleaned_students_dropout01.csv')

In [3]:
df

,School_Type,Location,Infrastructure,Teaching_Staff,Gender,Caste,Age,Standard,Socioeconomic_Status,Dropout_Status
0,Government,Semi-Urban,Poor,Poor,Male,SC,14,8,Low,Dropout
1,Private,Rural,Basic,Good,Female,General,15,7,High,Enrolled
2,Government,Rural,Good,Excellent,Male,ST,13,9,Moderate,Dropout
3,Private,Semi-Urban,Basic,Good,Female,OBC,12,10,High,Enrolled
4,Government,Rural,Basic,Poor,Male,SC,16,8,Low,Dropout
...,...,...,...,...,...,...,...,...,...,...
816,Government,Rural,Basic,Good,Female,ST,15,10,Moderate,Enrolled
817,Government,Rural,Poor,Poor,Male,SC,15,10,High,Enrolled
818,Government,Semi-Urban,Good,Good,Male,General,12,10,High,Enrolled
819,Private,Rural,Basic,Poor,Female,ST,11,9,Low,Dropout


In [4]:
df['Location'].unique()

array(['Semi-Urban', 'Rural', 'Urban'], dtype=object)

In [5]:
for col in df.columns:
    print(f"{col}: {df[col].unique()} unique values")

School_Type: ['Government' 'Private'] unique values
Location: ['Semi-Urban' 'Rural' 'Urban'] unique values
Infrastructure: ['Poor' 'Basic' 'Good' 'Excellent'] unique values
Teaching_Staff: ['Poor' 'Good' 'Excellent' 'Male' 'Female'] unique values
Gender: ['Male' 'Female'] unique values
Caste: ['SC' 'General' 'ST' 'OBC'] unique values
Age: [14 15 13 12 16 10 11] unique values
Standard: [ 8  7  9 10 12  5  6 11] unique values
Socioeconomic_Status: ['Low' 'High' 'Moderate'] unique values
Dropout_Status: ['Dropout' 'Enrolled'] unique values


In [6]:
dfCopy = df.copy()


In [7]:
X_train, X_test, y_train, y_test = train_test_split(df.drop('Dropout_Status', axis=1), df['Dropout_Status'], test_size=0.2, random_state=42)

In [8]:
dfCopy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 821 entries, 0 to 820
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   School_Type           821 non-null    object
 1   Location              821 non-null    object
 2   Infrastructure        821 non-null    object
 3   Teaching_Staff        821 non-null    object
 4   Gender                821 non-null    object
 5   Caste                 821 non-null    object
 6   Age                   821 non-null    int64 
 7   Standard              821 non-null    int64 
 8   Socioeconomic_Status  821 non-null    object
 9   Dropout_Status        821 non-null    object
dtypes: int64(2), object(8)
memory usage: 64.3+ KB


In [9]:
dfCopy.columns

Index(['School_Type', 'Location', 'Infrastructure', 'Teaching_Staff', 'Gender',
       'Caste', 'Age', 'Standard', 'Socioeconomic_Status', 'Dropout_Status'],
      dtype='object')

In [10]:
# cat_col = ['School_Type', 'Location', 'Infrastructure', 'Teaching_Staff', 'gender', 'Caste', 'Socioeconomic_Status']
num_col = ['Age', 'Standard']

In [11]:
Label_cat =['Dropout_Status'] 

In [12]:
dfCopy.columns

Index(['School_Type', 'Location', 'Infrastructure', 'Teaching_Staff', 'Gender',
       'Caste', 'Age', 'Standard', 'Socioeconomic_Status', 'Dropout_Status'],
      dtype='object')

In [13]:
onehot_cols = [
    'School_Type',
    'Location',
    'Gender',
    'Caste'
]
ordinal_cols = [
    'Infrastructure',
    'Teaching_Staff',
    'Socioeconomic_Status'
]

In [14]:
ordinal_categories = [
    ['Poor', 'Basic', 'Good', 'Excellent'],  # Infrastructure
    ['Poor', 'Good', 'Excellent', 'Male', 'Female'],  # Teaching_Staff
    ['Low', 'Moderate', 'High']  # Socioeconomic_Status
]

In [15]:
# cat_pipeline = Pipeline([
#     ('encoder', OneHotEncoder(handle_unknown='ignore'))
# ])
# num_pipeline = Pipeline([
#     ('scaler', StandardScaler())
# ])

In [16]:
from sklearn.preprocessing import OrdinalEncoder


num_pipeline = Pipeline([
    ('scaler', StandardScaler())
])

onehot_pipeline = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

ordinal_pipeline = Pipeline([
    ('ordinal', OrdinalEncoder(categories=ordinal_categories))
])
label_pipeline = Pipeline([
    ('label', LabelEncoder())
])


In [17]:
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_col),
    ('onehot', onehot_pipeline, onehot_cols),
    ('ordinal', ordinal_pipeline, ordinal_cols)
])

In [18]:
# from sklearn.linear_model import LogisticRegression


# model_pipeline = Pipeline([
#     ('preprocessing', preprocessor),
#     ('model', LogisticRegression(max_iter=1000))
# ])

In [19]:
# model_pipeline.fit(X_train, y_train)
# red = model_pipeline.predict(X_test)
# print(classification_report(y_test, red))
# print(confusion_matrix(y_test, red))


In [32]:
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV

rf_pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('model', RandomForestClassifier(max_depth=5, n_estimators=500, min_samples_split=20, min_samples_leaf=10, max_features='log2', random_state=42, class_weight='balanced'))
])
rf_pipeline.fit(X_train, y_train)
rf_pred = rf_pipeline.predict(X_test)
print(accuracy_score(y_test, rf_pred))
print(classification_report(y_test, rf_pred))
print(confusion_matrix(y_test, rf_pred))





0.9272727272727272
              precision    recall  f1-score   support

     Dropout       0.90      0.98      0.94        93
    Enrolled       0.97      0.86      0.91        72

    accuracy                           0.93       165
   macro avg       0.93      0.92      0.92       165
weighted avg       0.93      0.93      0.93       165

[[91  2]
 [10 62]]


In [ ]:
# from sklearn.model_selection import GridSearchCV
# param_grid = {
#     'model__n_estimators': [100, 200, 300],
#     'model__max_depth': [10, 20],
#     'model__min_samples_split': [2, 5],
#     'model__min_samples_leaf': [1, 2],
#     'model__max_features': ['sqrt', 'log2']
# }

# grid = GridSearchCV(
#     rf_pipeline,
#     param_grid=param_grid,
#     cv=5,
#     scoring='accuracy',
#     n_jobs=-1
# )

# grid.fit(X_train, y_train)
# best_model = grid.best_estimator_

# rf_pred = best_model.predict(X_test)
# print('Best Params:', grid.best_params_)
# print('Best CV Score:', round(grid.best_score_, 4))
# print(classification_report(y_test, rf_pred))
# print(confusion_matrix(y_test, rf_pred))

In [21]:
# ! pip install xgboost

In [33]:
import joblib
joblib.dump(rf_pipeline, "dropout_model.pkl")

['dropout_model.pkl']

In [34]:
! pip install shap
import joblib, shap

model = joblib.load("dropout_model.pkl")

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(sample)

print(dict(zip(sample.columns, shap_values[0][0])))

  Using cached slicer-0.0.8-py3-none-any.whl.metadata (4.0 kB)
   ---------------------------------------- 0.0/555.9 kB ? eta -:--:--
   ------------------------------------- -- 524.3/555.9 kB 5.3 MB/s eta 0:00:01
   ---------------------------------------- 555.9/555.9 kB 3.1 MB/s eta 0:00:00
Using cached slicer-0.0.8-py3-none-any.whl (15 kB)

   ---------------------------------------- 0/2 [slicer]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2

InvalidModelError: Model type not yet supported by TreeExplainer: <class 'sklearn.pipeline.Pipeline'>

In [37]:
pipe = joblib.load("dropout_model.pkl")

sample = pd.DataFrame([{
"School_Type": "Private",
"Location": "Semi-Urban",
"Infrastructure": "Good",
"Teaching_Staff": "Good",
"Gender": "Female",
"Caste": "General",
"Age": 14,
"Standard": 9,
"Socioeconomic_Status": "High"
}])

X_sample = pipe.named_steps["preprocessing"].transform(sample)
rf_model = pipe.named_steps["model"]

explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_sample)

feature_names = pipe.named_steps["preprocessing"].get_feature_names_out()

if isinstance(shap_values, list):
    # old SHAP format: list[class_idx] -> (n_samples, n_features)
    vals = shap_values[1][0] if len(shap_values) > 1 else shap_values[0][0]
else:
    # new SHAP format can be (n_samples, n_features, n_classes)
    if shap_values.ndim == 3:
        class_idx = 1 if shap_values.shape[2] > 1 else 0
        vals = shap_values[0, :, class_idx]
    else:
        vals = shap_values[0]

# print(dict(zip(feature_names, vals)))
contrib = dict(zip(feature_names, vals))

# sort by importance
sorted_contrib = sorted(contrib.items(), key=lambda x: abs(float(x[1])), reverse=True)

# print like your format
for f, v in sorted_contrib[:5]:
    print(f"{f}   {round(v,4)}")

ordinal__Socioeconomic_Status   0.1764
onehot__School_Type_Government   0.1117
onehot__School_Type_Private   0.0774
num__Standard   -0.0183
ordinal__Infrastructure   -0.0177
